In [6]:
import pandas as pd
from pathlib import Path

In [7]:
from pathlib import Path

input_folder = Path("/Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/02_ET_data")

# show a quick peek of files
all_files = sorted(input_folder.iterdir())
print("Total items in folder:", len(all_files))
print("First 20 items:")
for f in all_files[:20]:
    print(" -", f.name)

Total items in folder: 199
First 20 items:
 - .DS_Store
 - Sub129_ 01-12-25 09h36m.csv
 - sub001_11-11-25 13h40m.csv
 - sub002_11-11-25 13h40m.csv
 - sub003_11-11-25 14h32m.csv
 - sub004_11-11-25 14h32m.csv
 - sub005_11-11-25 15h09m.csv
 - sub006_11-11-25 15h10m.csv
 - sub007_11-11-25 15h51m.csv
 - sub008_11-11-25 16h43m.csv
 - sub009_11-11-25 16h42m.csv
 - sub010_11-11-25 17h39m.csv
 - sub011_11-11-25 17h43m.csv
 - sub012_12-11-25 10h58m.csv
 - sub013_12-11-25 10h58m.csv
 - sub014_12-11-25 11h52m.csv
 - sub015_12-11-25 12h36m.csv
 - sub016_12-11-25 13h06m.csv
 - sub019_12-11-25 13h48m.csv
 - sub020_12-11-25 14h49m.csv


In [8]:
import pandas as pd
from pathlib import Path

def extract_respondent_name_from_lines(lines):
    for line in lines:
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 2 and parts[0] == "#Respondent Name":
            return parts[1]
    return None

def find_data_header_row_from_lines(lines):
    # Header row is the one that has BOTH "Row" and "SourceStimuliName"
    for i, line in enumerate(lines):
        if "Row" in line and "SourceStimuliName" in line:
            return i
    return None

def extract_source_stimuli_from_csv(file_path: Path):
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    respondent = extract_respondent_name_from_lines(lines) or file_path.stem

    header_row = find_data_header_row_from_lines(lines)
    if header_row is None:
        return respondent, ""

    df = pd.read_csv(file_path, skiprows=header_row)
    df.columns = [str(c).strip() for c in df.columns]

    if "SourceStimuliName" not in df.columns:
        return respondent, ""

    stimuli = (
        df["SourceStimuliName"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # unique, preserve order
    seen = set()
    uniq = []
    for s in stimuli:
        if s and s not in seen:
            seen.add(s)
            uniq.append(s)

    return respondent, "; ".join(uniq)


In [9]:
input_folder = Path("/Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/02_ET_data")
output_xlsx = Path("/Users/hjcho/Documents/GitHub/PC_HJ/data/respondent_source_stimuli.xlsx")

files = sorted(input_folder.glob("*.csv"))
print(f"Found {len(files)} CSV files")

rows = []
for fp in files:
    # skip junk files if any
    if fp.name.startswith("._") or fp.name == ".DS_Store":
        continue

    try:
        respondent, source_stimuli = extract_source_stimuli_from_csv(fp)
        rows.append({
            "RespondentName": respondent,
            "SourceStimulus": source_stimuli
        })
    except Exception as e:
        print(f"[WARN] Failed on {fp.name}: {e}")
        rows.append({
            "RespondentName": fp.stem,
            "SourceStimulus": ""
        })

out_df = pd.DataFrame(rows)
out_df.to_excel(output_xlsx, index=False)

out_df.head(), f"Saved to → {output_xlsx}"


Found 198 CSV files


(            RespondentName SourceStimulus
 0  Sub129_ 01-12-25 09h36m            PCM
 1   sub001_11-11-25 13h40m            PCM
 2   sub002_11-11-25 13h40m             PC
 3   sub003_11-11-25 14h32m             PC
 4   sub004_11-11-25 14h32m             PC,
 'Saved to → /Users/hjcho/Documents/GitHub/PC_HJ/data/respondent_source_stimuli.xlsx')

In [10]:
test_file = Path("/Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/02_ET_data/sub009_11-11-25 16h42m.csv")
extract_source_stimuli_from_csv(test_file)

('sub009_11-11-25 16h42m', 'PC')

In [11]:
import pandas as pd
from pathlib import Path

base_folder = Path("/Users/hjcho/Documents/GitHub/PC_HJ/data")
raw_folder  = base_folder / "00_raw_data/02_ET_data"

summary_path = base_folder / "respondent_source_stimuli.xlsx"

summary_df = pd.read_excel(summary_path)
summary_df.head()


,RespondentName,SourceStimulus
0,Sub129_ 01-12-25 09h36m,PCM
1,sub001_11-11-25 13h40m,PCM
2,sub002_11-11-25 13h40m,PC
3,sub003_11-11-25 14h32m,PC
4,sub004_11-11-25 14h32m,PC


In [12]:
def extract_respondent_name_from_lines(lines):
    for line in lines:
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 2 and parts[0] == "#Respondent Name":
            return parts[1]
    return None

def find_data_header_row_from_lines(lines):
    for i, line in enumerate(lines):
        if "Row" in line and "SourceStimuliName" in line:
            return i
    return None

def extract_from_csv(file_path: Path):
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    respondent = extract_respondent_name_from_lines(lines) or file_path.stem

    header_row = find_data_header_row_from_lines(lines)
    if header_row is None:
        return respondent, ""

    df = pd.read_csv(file_path, skiprows=header_row)
    df.columns = [str(c).strip() for c in df.columns]

    if "SourceStimuliName" not in df.columns:
        return respondent, ""

    stimuli = (
        df["SourceStimuliName"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    seen = set()
    uniq = []
    for s in stimuli:
        if s and s not in seen:
            seen.add(s)
            uniq.append(s)

    return respondent, "; ".join(uniq)


In [13]:
csv_files = sorted(raw_folder.glob("*.csv"))
print("CSV files found:", len(csv_files))

truth_rows = []

for fp in csv_files:
    if fp.name.startswith("._") or fp.name == ".DS_Store":
        continue

    respondent, stimuli = extract_from_csv(fp)

    truth_rows.append({
        "RespondentName": respondent,
        "SourceStimulus_extracted": stimuli,
        "SourceFile": fp.name
    })

truth_df = pd.DataFrame(truth_rows)
truth_df.head()


CSV files found: 198


,RespondentName,SourceStimulus_extracted,SourceFile
0,Sub129_ 01-12-25 09h36m,PCM,Sub129_ 01-12-25 09h36m.csv
1,sub001_11-11-25 13h40m,PCM,sub001_11-11-25 13h40m.csv
2,sub002_11-11-25 13h40m,PC,sub002_11-11-25 13h40m.csv
3,sub003_11-11-25 14h32m,PC,sub003_11-11-25 14h32m.csv
4,sub004_11-11-25 14h32m,PC,sub004_11-11-25 14h32m.csv


In [14]:
check_df = summary_df.merge(
    truth_df,
    on="RespondentName",
    how="outer"
)

# Normalize strings for comparison
def normalize(x):
    if pd.isna(x):
        return ""
    return "; ".join(sorted([s.strip() for s in str(x).split(";") if s.strip()]))

check_df["SourceStimulus_summary_norm"] = check_df["SourceStimulus"].apply(normalize)
check_df["SourceStimulus_extracted_norm"] = check_df["SourceStimulus_extracted"].apply(normalize)

check_df["MATCH"] = (
    check_df["SourceStimulus_summary_norm"]
    == check_df["SourceStimulus_extracted_norm"]
)

check_df.head()


,RespondentName,SourceStimulus,SourceStimulus_extracted,SourceFile,SourceStimulus_summary_norm,SourceStimulus_extracted_norm,MATCH
0,Sub129_ 01-12-25 09h36m,PCM,PCM,Sub129_ 01-12-25 09h36m.csv,PCM,PCM,True
1,sub001_11-11-25 13h40m,PCM,PCM,sub001_11-11-25 13h40m.csv,PCM,PCM,True
2,sub002_11-11-25 13h40m,PC,PC,sub002_11-11-25 13h40m.csv,PC,PC,True
3,sub003_11-11-25 14h32m,PC,PC,sub003_11-11-25 14h32m.csv,PC,PC,True
4,sub004_11-11-25 14h32m,PC,PC,sub004_11-11-25 14h32m.csv,PC,PC,True


In [15]:
mismatches = check_df[check_df["MATCH"] == False]

print("Total rows:", len(check_df))
print("Mismatches:", len(mismatches))

mismatches[
    [
        "RespondentName",
        "SourceStimulus",
        "SourceStimulus_extracted",
        "SourceFile"
    ]
].head(10)


Total rows: 198
Mismatches: 0


,RespondentName,SourceStimulus,SourceStimulus_extracted,SourceFile


In [16]:
verification_path = base_folder / "respondent_source_stimuli_VERIFICATION.xlsx"
check_df.to_excel(verification_path, index=False)

verification_path


PosixPath('/Users/hjcho/Documents/GitHub/PC_HJ/data/respondent_source_stimuli_VERIFICATION.xlsx')